In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib


In [2]:
balls = pd.read_csv("Downloads/IPL_Project/deliveries.csv")
matches = pd.read_csv("Downloads/IPL_Project/matches.csv")

In [3]:
balls = balls.rename(columns={
    'id':'match_id',
    'batsman_run':'batsman_runs',
    'iswicket_delivery':'is_wicket',
    'player_out':'player_dismissed',
    'dismisal_kind':'dismissal_kind',
    'overs':'over',
    'ball_number':'ball'
})

matches = matches.rename(columns={
    'id':'match_id',
    'match_date':'date'
})

matches['date'] = pd.to_datetime(matches['date'], dayfirst=True, errors='coerce')


In [4]:
match_cols = ['match_id','date','venue','team1','team2']
df = balls.merge(matches[match_cols], on='match_id', how='left')
df = df.sort_values('date')


In [5]:
# =====================================
# CREATE BOWLING TEAM COLUMN
# =====================================

df['bowling_team'] = np.where(
    df['batting_team'] == df['team1'],
    df['team2'],
    df['team1']
)

print("bowling_team created")
df[['batting_team','team1','team2','bowling_team']].head()


bowling_team created


,batting_team,team1,team2,bowling_team
225784,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore
225810,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore
225809,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore
225808,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore
225807,Kolkata Knight Riders,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore


In [6]:
bat_match = (
    df.groupby(['match_id','date','venue','batting_team','bowling_team','batter'])
      .agg(
          runs=('batsman_runs','sum'),
          balls=('ball','count'),
          fours=('batsman_runs', lambda x:(x==4).sum()),
          sixes=('batsman_runs', lambda x:(x==6).sum())
      )
      .reset_index()
)

bat_match['strike_rate'] = (bat_match['runs']/bat_match['balls'])*100


In [7]:
bowl_match = (
    df.groupby(['match_id','date','venue','batting_team','bowling_team','bowler'])
      .agg(
          balls=('ball','count'),
          runs_conceded=('total_run','sum'),
          wickets=('is_wicket','sum')
      )
      .reset_index()
)

bowl_match['overs'] = bowl_match['balls']/6
bowl_match['economy'] = bowl_match['runs_conceded']/bowl_match['overs']


In [8]:
bat_match = bat_match.sort_values(['batter','date'])

bat_match['form_5'] = bat_match.groupby('batter')['runs'].shift().rolling(5,min_periods=1).mean()
bat_match['form_10'] = bat_match.groupby('batter')['runs'].shift().rolling(10,min_periods=1).mean()


In [9]:
bowl_match = bowl_match.sort_values(['bowler','date'])

bowl_match['form_5'] = bowl_match.groupby('bowler')['wickets'].shift().rolling(5,min_periods=1).mean()
bowl_match['form_10'] = bowl_match.groupby('bowler')['wickets'].shift().rolling(10,min_periods=1).mean()


In [10]:
bat_match['venue_avg'] = bat_match.groupby(['batter','venue'])['runs'].transform(lambda x: x.shift().expanding().mean())
bowl_match['venue_avg'] = bowl_match.groupby(['bowler','venue'])['wickets'].transform(lambda x: x.shift().expanding().mean())


In [11]:
bat_match['vs_team'] = bat_match.groupby(['batter','bowling_team'])['runs'].transform(lambda x: x.shift().expanding().mean())
bowl_match['vs_team'] = bowl_match.groupby(['bowler','batting_team'])['wickets'].transform(lambda x: x.shift().expanding().mean())


In [12]:
bat_match['target_runs'] = bat_match.groupby('batter')['runs'].shift(-1)
bowl_match['target_wkts'] = bowl_match.groupby('bowler')['wickets'].shift(-1)


In [13]:
bat_final = bat_match.dropna()
bowl_final = bowl_match.dropna()


In [14]:
split_date = '2022-01-01'

bat_train = bat_final[bat_final['date'] < split_date]
bat_test  = bat_final[bat_final['date'] >= split_date]

bowl_train = bowl_final[bowl_final['date'] < split_date]
bowl_test  = bowl_final[bowl_final['date'] >= split_date]


In [15]:
bat_final.to_csv("batting_dataset.csv", index=False)
bowl_final.to_csv("bowling_dataset.csv", index=False)


In [16]:
num_features = ['form_5','form_10','venue_avg','vs_team','career_avg','strike_rate']

pipeline = ColumnTransformer([
    ('num', StandardScaler(), num_features)
])

joblib.dump(pipeline, "feature_pipeline.pkl")


['feature_pipeline.pkl']